# Player Aggregation

Sanity-check the per-player feature table built by
`archetypes.player_aggregation.build_player_feature_table`.

Each row = one player. Feature columns are, for each event type in
the player's group:

- `pct_total_<event_type>` - share of player's events of this type
- `pct_cluster_<event_type>_<k>` - within that event type, share in cluster `k`

The `pct_cluster_*` columns within a single event type should sum
to ≈1 for every player; the `pct_total_*` columns across event
types should also sum to ≈1.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# locate repo root from the notebook's location
HERE = Path.cwd()
ROOT = HERE
while ROOT != ROOT.parent and not (ROOT / "configs" / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

CFG = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
ART = ROOT / CFG["data_paths"]["archetype_artifacts"]
EVENT_MODEL_DIR = ART / "event_clusters"
ARCH_MODEL_DIR = ART / "archetype_clusters"

EVENT_TYPES_PER_GROUP = CFG["event_clustering"]["event_types_per_group"]
N_CLUSTERS_PER_EVENT = CFG["event_clustering"]["n_clusters_per_event"]
N_ARCHETYPES_PER_GROUP = CFG["archetype_clustering"]["n_archetypes_per_group"]
POSITION_GROUPS = list(EVENT_TYPES_PER_GROUP.keys())

print("ROOT:", ROOT)
print("ART :", ART)

In [ ]:
feats = pd.read_parquet(ART / "player_features.parquet")
print(f"player features: {feats.shape}")
feats.head()

## Player counts per position group

In [ ]:
counts = feats["position_group"].value_counts().reindex(POSITION_GROUPS)
fig, ax = plt.subplots(figsize=(6, 3))
counts.plot.bar(ax=ax, color="steelblue")
ax.set_ylabel("# players")
ax.set_title("Players per position group (after min-events filter)")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## Aggregation invariants

For each player, the within-event-type cluster shares should sum
to 1 (or 0 if they had no events of that type), and the cross-type
`pct_total_*` shares should sum to 1.

In [ ]:
def invariant_summary(feats):
    rows = []
    for group, ets in EVENT_TYPES_PER_GROUP.items():
        sub = feats[feats["position_group"] == group]
        total_sum = sub[[f"pct_total_{et}" for et in ets]].sum(axis=1)
        rows.append({"group": group, "check": "pct_total_* sum",
                     "min": total_sum.min().round(4),
                     "max": total_sum.max().round(4)})
        for et in ets:
            cols = [f"pct_cluster_{et}_{i}"
                    for i in range(N_CLUSTERS_PER_EVENT[group][et])]
            s = sub[cols].sum(axis=1)
            # 0 (no events of type) or 1 (had events) is the expected pattern
            rows.append({"group": group, "check": f"cluster_{et} sum",
                         "min": s.min().round(4),
                         "max": s.max().round(4)})
    return pd.DataFrame(rows)

invariant_summary(feats)

## Distribution of `pct_total_<event_type>` by position group

In [ ]:
for group, ets in EVENT_TYPES_PER_GROUP.items():
    sub = feats[feats["position_group"] == group]
    cols = [f"pct_total_{et}" for et in ets]
    fig, ax = plt.subplots(figsize=(1.5 + 1.5 * len(cols), 3))
    sub[cols].plot.box(ax=ax, grid=True)
    ax.set_xticklabels([c.replace("pct_total_", "") for c in cols], rotation=20)
    ax.set_ylabel("share of player's events")
    ax.set_title(f"{group}: event-type mix (n={len(sub)})")
    plt.tight_layout()
    plt.show()

## Mean feature heatmap (averaged within each position group)

In [ ]:
mean_by_group = feats.groupby("position_group").mean(numeric_only=True)
mean_by_group = mean_by_group.drop(columns=["player_id"], errors="ignore")
mean_by_group = mean_by_group.reindex(POSITION_GROUPS)

fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(mean_by_group.columns))))
im = ax.imshow(mean_by_group.T.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(mean_by_group.index)))
ax.set_xticklabels(mean_by_group.index)
ax.set_yticks(range(len(mean_by_group.columns)))
ax.set_yticklabels(mean_by_group.columns, fontsize=8)
fig.colorbar(im, ax=ax, label="mean value")
ax.set_title("Mean feature value per position group")
plt.tight_layout()
plt.show()

## Example player vectors

The five players with the most events in each group - useful for
eyeballing whether the feature layout makes sense.

In [ ]:
# we don't store raw event counts on the feature table itself;
# players are listed in order of insertion (groupby), so just show
# a few random examples per group.
for group in POSITION_GROUPS:
    sub = feats[feats["position_group"] == group]
    print(f"--- {group} (showing 5 random players of {len(sub)}) ---")
    print(sub.sample(min(5, len(sub)), random_state=0).round(3).to_string(index=False))
    print()